In [1]:
# ===== Contextual Compression Demo (Colab one cell, no API key) =====
!pip -q install -U sentence-transformers chromadb

import re
import chromadb
from sentence_transformers import SentenceTransformer
from typing import List, Tuple

# -----------------------------
# 1) Sample "retrieved chunks" corpus (pretend these are PDF chunks)
# -----------------------------
docs = [
("d1", """Add/Drop is allowed in Week 1 and Week 2.
Students should check the academic calendar.
Late add/drop requests require approval.
The university also provides counselling services for study planning."""),

("d2", """A prerequisite is a module you must complete before enrolling in another module.
Some modules also have co-requisites.
The module outline lists prerequisites under the requirements section."""),

("d3", """Tuition fees depend on citizenship and programme.
Payment can be made by GIRO or credit card.
Late payment may incur penalties.
Scholarship details are on a separate website."""),

("d4", """If you are working part-time, allocate fixed study blocks each week.
Prioritize assessments with higher weightage.
Keep at least one buffer day before deadlines.
Use a weekly review routine to adjust your plan."""),

("d5", """The academic calendar includes exam periods and public holidays.
Some dates may change, so always verify on the official portal.
Students can contact the school office for clarification."""),
]

# -----------------------------
# 2) Build vector store + retriever
# -----------------------------
embed = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.Client()
col = client.get_or_create_collection("compression_demo")

ids = [i for i,_ in docs]
texts = [t for _,t in docs]
embs = embed.encode(texts, normalize_embeddings=True).tolist()

try:
    col.delete(ids=ids)
except Exception:
    pass
col.add(ids=ids, documents=texts, embeddings=embs)

def retrieve_top_k(query: str, k: int = 3) -> List[Tuple[str,str]]:
    q_emb = embed.encode([query], normalize_embeddings=True).tolist()
    res = col.query(query_embeddings=q_emb, n_results=k)
    return list(zip(res["ids"][0], res["documents"][0]))

# -----------------------------
# 3) Contextual compression (sentence extraction)
#    Keep only sentences most similar to the query
# -----------------------------
def split_sentences(text: str) -> List[str]:
    # simple sentence splitter for teaching
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    return [s.strip() for s in sents if s.strip()]

def compress_chunk(query: str, chunk_text: str, keep_top_n_sentences: int = 2) -> str:
    sentences = split_sentences(chunk_text)
    if not sentences:
        return chunk_text

    # embed query + sentences, rank by cosine similarity (dot product because normalized)
    qv = embed.encode([query], normalize_embeddings=True)
    sv = embed.encode(sentences, normalize_embeddings=True)

    scores = (sv @ qv[0]).tolist()
    ranked = sorted(zip(sentences, scores), key=lambda x: x[1], reverse=True)[:keep_top_n_sentences]

    # return only best sentences (compressed)
    return " ".join([s for s,_ in ranked])

def contextual_compression(query: str, retrieved: List[Tuple[str,str]], keep_sentences: int = 2):
    compressed = []
    for doc_id, chunk in retrieved:
        comp = compress_chunk(query, chunk, keep_top_n_sentences=keep_sentences)
        compressed.append((doc_id, comp))
    return compressed

# -----------------------------
# 4) Demo run
# -----------------------------
query = "What is the add/drop deadline?"
retrieved = retrieve_top_k(query, k=3)
compressed = contextual_compression(query, retrieved, keep_sentences=2)

print("QUERY:", query)

print("\n--- Retrieved (top-k chunks) ---")
for i,(doc_id,txt) in enumerate(retrieved,1):
    print(f"\n{i}. {doc_id}:\n{txt}")

print("\n--- After Contextual Compression (only most relevant sentences) ---")
for i,(doc_id,txt) in enumerate(compressed,1):
    print(f"\n{i}. {doc_id}:\n{txt}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 67.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 88.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 86.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.6/132.6 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not current

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

QUERY: What is the add/drop deadline?

--- Retrieved (top-k chunks) ---

1. d1:
Add/Drop is allowed in Week 1 and Week 2.
Students should check the academic calendar.
Late add/drop requests require approval.
The university also provides counselling services for study planning.

2. d5:
The academic calendar includes exam periods and public holidays.
Some dates may change, so always verify on the official portal.
Students can contact the school office for clarification.

3. d4:
If you are working part-time, allocate fixed study blocks each week.
Prioritize assessments with higher weightage.
Keep at least one buffer day before deadlines.
Use a weekly review routine to adjust your plan.

--- After Contextual Compression (only most relevant sentences) ---

1. d1:
Late add/drop requests require approval. Add/Drop is allowed in Week 1 and Week 2.

2. d5:
Some dates may change, so always verify on the official portal. The academic calendar includes exam periods and public holidays.

3. d4:
Kee